# CWA 資料探索（課程第 5–7、10、12 步）

用 pandas 觀察氣象署 JSON 的結構與整理後的資料庫內容。網站本身不使用 pandas；這份 notebook 是資料探索與驗證用。

先執行 `python -m scripts.fetch_samples`、`python -m scripts.init_db`、`python -m scripts.run_job all`。

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sample = lambda name: json.loads((ROOT / 'tests' / 'samples' / f'{name}.json').read_text(encoding='utf-8'))

## 第 5 步：JSON 結構

一週預報 `F-D0047-091` 的路徑：`records.Locations[0].Location[].WeatherElement[].Time[]`。

In [ ]:
week = sample('F-D0047-091')
location = week['records']['Locations'][0]['Location'][0]
print(location['LocationName'])
[e['ElementName'] for e in location['WeatherElement']]

## 第 6 步：取出最高與最低溫

In [ ]:
rows = []
for loc in week['records']['Locations'][0]['Location']:
    for element in loc['WeatherElement']:
        if element['ElementName'] in ('最高溫度', '最低溫度'):
            for t in element['Time']:
                value = list(t['ElementValue'][0].values())[0]
                rows.append({'regionName': loc['LocationName'], 'element': element['ElementName'],
                             'startTime': t['StartTime'], 'value': float(value)})
raw = pd.DataFrame(rows)
raw.head()

## 第 7 步：用 pandas 整理與預覽

In [ ]:
wide = raw.pivot_table(index=['regionName', 'startTime'], columns='element', values='value').reset_index()
wide.columns.name = None
wide = wide.rename(columns={'最低溫度': 'mint', '最高溫度': 'maxt'})
print(wide.shape)
wide.describe()

In [ ]:
assert (wide['mint'] <= wide['maxt']).all()
wide[wide['regionName'] == '臺中市']

## 第 10、12 步：從 SQLite 讀取並驗證

`TemperatureForecasts` 保存每一版預報；`LatestTemperatureForecasts` 是每個縣市、每一天的最新一版。

In [ ]:
conn = sqlite3.connect(ROOT / 'data' / 'weather.db')
df = pd.read_sql_query('SELECT regionName, dataDate, mint, maxt, pop, approx FROM LatestTemperatureForecasts ORDER BY regionName, dataDate', conn)
df.head(10)

In [ ]:
pd.read_sql_query('SELECT DISTINCT regionName FROM TemperatureForecasts', conn)

In [ ]:
pd.read_sql_query("SELECT * FROM LatestTemperatureForecasts WHERE regionName = '臺中市'", conn)

In [ ]:
obs = pd.read_sql_query('SELECT county, observedAt, temperature, humidity, pressure FROM CountyObservations ORDER BY observedAt DESC LIMIT 22', conn)
obs.sort_values('temperature', ascending=False)